In [41]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, date
from pandas.core.config_init import register_converter_doc
import zipfile

from sklearn.metrics.pairwise import haversine_distances
from math import radians
from tqdm import tqdm

In [34]:
DATE_PLUS_MINUS_DAYS = 30
RADIUS_DIST_KM = 100.0 # KMs

In [20]:
def make_dataframe_cols_lowercase(df):
    for col in df.columns:
        df.rename(columns={col: col.lower()}, inplace=True)

def add_formatted_join_column(df, join_col_name, drop_cols=None):
    df['sample date (date)'] = pd.to_datetime(df['sample date'], format='%d-%m-%Y').dt.date
    df['sample date'] = pd.to_datetime(df['sample date'], format='%d-%m-%Y').dt.strftime('%d-%m-%Y')
    df['latitude'] = df['latitude'].astype(float).round(6)
    df['longitude'] = df['longitude'].astype(float).round(6)
    df[join_col_name] = df['latitude'].astype(str) + "~" + df['longitude'].astype(str) + "~" + df['sample date']

    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

In [21]:
wq_data_start_date = date(2010, 1, 1)
wq_data_end_date = date(2016, 12, 31)

In [22]:
wq_df = pd.read_csv("../../water_quality_training_dataset.csv")
make_dataframe_cols_lowercase(wq_df)
add_formatted_join_column(wq_df, "join_column")
wq_df['sample date plus'] = wq_df['sample date (date)'] + pd.Timedelta(days=DATE_PLUS_MINUS_DAYS)
wq_df['sample date minus'] = wq_df['sample date (date)'] - pd.Timedelta(days=DATE_PLUS_MINUS_DAYS)

In [23]:
df_temp = pd.read_csv("temperature_degC_raw.csv")
df_ox = pd.read_csv("oxygen_mgLO2_raw.csv")
df_chl = pd.read_csv("chl.csv")
df_tp = pd.read_csv("tp.csv")

In [ ]:
sa_wq_df = pd.read_csv("new_water_quality_data/extracted_data/extracted_depWatSanData.csv")

In [ ]:
df_monitoring_feature_locations = sa_wq_df[["data_id", "latitude", "longitude"]].drop_duplicates()

In [26]:
df_monitoring_feature_locations["data_id_1"] = df_monitoring_feature_locations["data_id"].str.split(" ").str[0]
df_monitoring_feature_locations["data_id_2"] = df_monitoring_feature_locations["data_id"].str.split(" ").str[1].astype(int)

In [27]:
df_list = [df_temp, df_ox, df_chl, df_tp]

for index, df in enumerate(df_list):
    format_string = "%Y-%m-%d"
    df["sample date (date)"] = pd.to_datetime(df["sample_begin_date"], format=format_string).dt.date

    df = df[df["sample date (date)"] >= wq_data_start_date]
    df = df[df["sample date (date)"] <= wq_data_end_date]

    df = df.merge(df_monitoring_feature_locations, left_on="mon_feature_id", right_on="data_id_2", how="inner")

    df_list[index] = df


In [52]:
sa_wq_df = df_list[3]
count = 0
for index, wq_row in tqdm(wq_df.iterrows(), desc="Process Water Quality Data"):
# wq_row = wq_df.iloc[6]

    lat = wq_row["latitude"]
    lon = wq_row["longitude"]
    sample_date = wq_row["sample date"]
    sample_date_plus = wq_row["sample date plus"]
    sample_date_minus = wq_row["sample date minus"]

    sa_wq_df_filter = sa_wq_df[(sample_date_minus <= sa_wq_df["sample date (date)"]) & (sa_wq_df["sample date (date)"] <= sample_date_plus)]
    if sa_wq_df_filter.shape[0] == 0:
        continue
    sa_wq_df_filter['euc_dist_dec_deg'] = np.sqrt((sa_wq_df_filter['latitude'] - lat)**2 + (sa_wq_df_filter['longitude'] - lon)**2)
    sa_wq_df_filter = sa_wq_df_filter.sort_values(by='euc_dist_dec_deg')
    sa_wq_df_filter['latitude_rad'] = np.radians(sa_wq_df_filter['latitude'])
    sa_wq_df_filter['longitude_rad'] = np.radians(sa_wq_df_filter['longitude'])

    ref_point_rad = [radians(lat), radians(lon)]

    sa_wq_df_filter['euc_dist_km'] = haversine_distances(sa_wq_df_filter[['latitude_rad', 'longitude_rad']], [ref_point_rad]) * 6371
    sa_wq_df_filter = sa_wq_df_filter.sort_values(by='euc_dist_km')
    sa_wq_df_filter = sa_wq_df_filter[sa_wq_df_filter['euc_dist_km'] < RADIUS_DIST_KM]

    if sa_wq_df_filter.shape[0] > 0:
        count += 1
# sa_wq_df_filter

Process Water Quality Data: 9319it [00:26, 355.00it/s]


In [53]:
count

2350